In [1]:
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pytz
import datetime as dt

try:
    import snowflake.connector
except:
    ! pip install snowflake 
    import snowflake.connector

import snowflake.connector
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa, dsa
from cryptography.hazmat.primitives import serialization

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2025-02-07 16:27:07.437295


#### Functions

#### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

# subtask
str_subtask = os.getcwd().split('/')[6]
print(f'Subtask: {str_subtask}')

str_dirname_output = './output'

str_date_min = '2024-11-26'

str_date_max = '2025-01-21'

str_date_min_2 = str_date_max

str_date_max_2 = '2025-02-07'

Project: 20241112-simple-model-test
Task: parse_all_apps
Subtask: 03_pull_data_dl


#### Connect

In [4]:
# load 
str_filename = 'datascience_rsa_key.p8'
str_local_path = f'./{str_filename}'
with open(str_local_path, "rb") as key:
    p_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend(),
    )

# convert to bytes
private_key = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

# connect to snowflake
conn = snowflake.connector.connect(
    user='datascience', 
    private_key=private_key,
    account='pfs', 
    warehouse='datascience',
    database='raw',
    schema='source_s3_scorehistory',
)

#### Query

In [5]:
# query
str_query = f"""
WITH RankedPayloads AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY accountid ORDER BY request_datetime DESC) AS rn
    FROM raw.source_s3_scorehistory.PAYLOAD_PARSED
    WHERE DATE(request_datetime) >= '{str_date_min}' 
    AND DATE(request_datetime) < '{str_date_max}'
    AND response_model_name = 'PRESTIGE-GEN-XII'
)
SELECT *
FROM RankedPayloads
WHERE rn = 1;
"""
print(str_query)


WITH RankedPayloads AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY accountid ORDER BY request_datetime DESC) AS rn
    FROM raw.source_s3_scorehistory.PAYLOAD_PARSED
    WHERE DATE(request_datetime) >= '2024-11-26' 
    AND DATE(request_datetime) < '2025-01-21'
    AND response_model_name = 'PRESTIGE-GEN-XII'
)
SELECT *
FROM RankedPayloads
WHERE rn = 1;



#### Pull data

In [6]:
%%time

# pull payloads
df = pd.read_sql(
    sql=str_query,
    con=conn,
)
# show
df

<timed exec>:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


CPU times: user 4min 17s, sys: 1min 56s, total: 6min 13s
Wall time: 3min 33s


,FILE_NAME,ACCOUNTID,REQUEST_DATETIME,FILE_VALUE,REQUEST_JSON,RESPONSE_JSON,REQUEST_ID,RESPONSE_MODEL_NAME,RESPONSE_MODEL_VERSION,RN
0,8544966/x3QS-2025-01-06-14:38:36,8544966,2025-01-06 21:38:36+00:00,"{\n ""request"": {\n ""request_id"": ""85449668...","{\n ""request_id"": ""8544966884627"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8544966884627,PRESTIGE-GEN-XII,V1,1
1,8506900/Z92q-2024-12-18-19:14:55,8506900,2024-12-19 02:14:55+00:00,"{\n ""request"": {\n ""request_id"": ""85069004...","{\n ""request_id"": ""8506900413352"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8506900413352,PRESTIGE-GEN-XII,V1,1
2,8561362/4Ln9-2025-01-13-18:49:00,8561362,2025-01-14 01:49:00+00:00,"{\n ""request"": {\n ""request_id"": ""85613627...","{\n ""request_id"": ""85613627021"",\n ""rows"": [...","{\n ""Response"": [\n {\n ""CounterOffer...",85613627021,PRESTIGE-GEN-XII,V1,1
3,8474876/Nsi8-2024-12-09-21:37:38,8474876,2024-12-10 04:37:38+00:00,"{\n ""request"": {\n ""request_id"": ""84748766...","{\n ""request_id"": ""8474876689953"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8474876689953,PRESTIGE-GEN-XII,V1,1
4,8465170/c69Z-2024-12-05-17:24:35,8465170,2024-12-06 00:24:35+00:00,"{\n ""request"": {\n ""request_id"": ""84651705...","{\n ""request_id"": ""8465170579834"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8465170579834,PRESTIGE-GEN-XII,V1,1
...,...,...,...,...,...,...,...,...,...,...
88657,8469056/3lgV-2024-12-06-21:30:33,8469056,2024-12-07 04:30:33+00:00,"{\n ""request"": {\n ""request_id"": ""84690563...","{\n ""request_id"": ""8469056369541"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8469056369541,PRESTIGE-GEN-XII,V1,1
88658,8560281/i571-2025-01-11-22:30:42,8560281,2025-01-12 05:30:42+00:00,"{\n ""request"": {\n ""request_id"": ""85602816...","{\n ""request_id"": ""8560281603978"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8560281603978,PRESTIGE-GEN-XII,V1,1
88659,8506082/Gy54-2024-12-18-15:59:13,8506082,2024-12-18 22:59:13+00:00,"{\n ""request"": {\n ""request_id"": ""85060823...","{\n ""request_id"": ""8506082398019"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8506082398019,PRESTIGE-GEN-XII,V1,1
88660,8554968/8FHE-2025-01-09-23:57:37,8554968,2025-01-10 06:57:37+00:00,"{\n ""request"": {\n ""request_id"": ""85549685...","{\n ""request_id"": ""8554968590809"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8554968590809,PRESTIGE-GEN-XII,V1,1


#### Query for DLv1 accounts

In [7]:
str_query = f"""
SELECT ACCOUNTID
FROM 
    raw.source_s3_scorehistory.PAYLOAD_PARSED
WHERE DATE(request_datetime) >= '{str_date_min}' 
AND DATE(request_datetime) < '{str_date_max}'
AND response_model_name = 'PRESTIGE-DLV1'
"""

#### Pull data

In [8]:
%%time

# pull payloads
df_tmp = pd.read_sql(
    sql=str_query,
    con=conn,
)
# get list of accounts
list_accountid = list(df_tmp['ACCOUNTID'])
# save memory
del df_tmp

<timed exec>:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


CPU times: user 70.6 ms, sys: 957 μs, total: 71.5 ms
Wall time: 304 ms


#### Use only DLv1 accounts

In [9]:
df = df[df['ACCOUNTID'].isin(list_accountid)].copy()
# show
df

,FILE_NAME,ACCOUNTID,REQUEST_DATETIME,FILE_VALUE,REQUEST_JSON,RESPONSE_JSON,REQUEST_ID,RESPONSE_MODEL_NAME,RESPONSE_MODEL_VERSION,RN
14,8516515/cJ2b-2024-12-22-12:43:29,8516515,2024-12-22 19:43:29+00:00,"{\n ""request"": {\n ""request_id"": ""85165158...","{\n ""request_id"": ""85165158801"",\n ""rows"": [...","{\n ""Response"": [\n {\n ""CounterOffer...",85165158801,PRESTIGE-GEN-XII,V1,1
20,8546549/j5Jv-2025-01-06-21:48:36,8546549,2025-01-07 04:48:36+00:00,"{\n ""request"": {\n ""request_id"": ""85465495...","{\n ""request_id"": ""8546549576153"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8546549576153,PRESTIGE-GEN-XII,V1,1
38,8474541/d1Pb-2024-12-09-19:15:09,8474541,2024-12-10 02:15:09+00:00,"{\n ""request"": {\n ""request_id"": ""84745411...","{\n ""request_id"": ""8474541141607"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8474541141607,PRESTIGE-GEN-XII,V1,1
45,8473303/7Kcq-2024-12-09-15:27:16,8473303,2024-12-09 22:27:16+00:00,"{\n ""request"": {\n ""request_id"": ""84733037...","{\n ""request_id"": ""8473303742301"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8473303742301,PRESTIGE-GEN-XII,V1,1
50,8519243/734R-2024-12-24-01:34:49,8519243,2024-12-24 08:34:49+00:00,"{\n ""request"": {\n ""request_id"": ""85192436...","{\n ""request_id"": ""8519243666180"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8519243666180,PRESTIGE-GEN-XII,V1,1
...,...,...,...,...,...,...,...,...,...,...
88627,8511808/ivJZ-2024-12-20-17:41:51,8511808,2024-12-21 00:41:51+00:00,"{\n ""request"": {\n ""request_id"": ""85118085...","{\n ""request_id"": ""8511808560159"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8511808560159,PRESTIGE-GEN-XII,V1,1
88637,8484906/DMah-2024-12-13-17:53:59,8484906,2024-12-14 00:53:59+00:00,"{\n ""request"": {\n ""request_id"": ""84849066...","{\n ""request_id"": ""8484906671646"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8484906671646,PRESTIGE-GEN-XII,V1,1
88639,8425394/7r0k-2024-11-26-01:57:40,8425394,2024-11-26 08:57:40+00:00,"{\n ""request"": {\n ""request_id"": ""84253948...","{\n ""request_id"": ""8425394898555"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8425394898555,PRESTIGE-GEN-XII,V1,1
88641,8430925/S98m-2024-11-29-16:54:11,8430925,2024-11-29 23:54:11+00:00,"{\n ""request"": {\n ""request_id"": ""84309251...","{\n ""request_id"": ""8430925150803"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8430925150803,PRESTIGE-GEN-XII,V1,1


#### Pull dlv1 apps only because they stopped background scoring gen 12

In [10]:
# query
str_query = f"""
WITH RankedPayloads AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY accountid ORDER BY request_datetime DESC) AS rn
    FROM raw.source_s3_scorehistory.PAYLOAD_PARSED
    WHERE DATE(request_datetime) >= '{str_date_min_2}' 
    AND DATE(request_datetime) < '{str_date_max_2}'
    AND response_model_name = 'PRESTIGE-DLV1'
)
SELECT *
FROM RankedPayloads
WHERE rn = 1;
"""
print(str_query)


WITH RankedPayloads AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY accountid ORDER BY request_datetime DESC) AS rn
    FROM raw.source_s3_scorehistory.PAYLOAD_PARSED
    WHERE DATE(request_datetime) >= '2025-01-21' 
    AND DATE(request_datetime) < '2025-02-07'
    AND response_model_name = 'PRESTIGE-DLV1'
)
SELECT *
FROM RankedPayloads
WHERE rn = 1;



In [11]:
%%time

# pull payloads
df_tmp = pd.read_sql(
    sql=str_query,
    con=conn,
)

<timed exec>:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


CPU times: user 13.5 s, sys: 4.55 s, total: 18 s
Wall time: 13.8 s


#### Concat

In [12]:
df = pd.concat([df, df_tmp])
# show
df

,FILE_NAME,ACCOUNTID,REQUEST_DATETIME,FILE_VALUE,REQUEST_JSON,RESPONSE_JSON,REQUEST_ID,RESPONSE_MODEL_NAME,RESPONSE_MODEL_VERSION,RN
14,8516515/cJ2b-2024-12-22-12:43:29,8516515,2024-12-22 19:43:29+00:00,"{\n ""request"": {\n ""request_id"": ""85165158...","{\n ""request_id"": ""85165158801"",\n ""rows"": [...","{\n ""Response"": [\n {\n ""CounterOffer...",85165158801,PRESTIGE-GEN-XII,V1,1
20,8546549/j5Jv-2025-01-06-21:48:36,8546549,2025-01-07 04:48:36+00:00,"{\n ""request"": {\n ""request_id"": ""85465495...","{\n ""request_id"": ""8546549576153"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8546549576153,PRESTIGE-GEN-XII,V1,1
38,8474541/d1Pb-2024-12-09-19:15:09,8474541,2024-12-10 02:15:09+00:00,"{\n ""request"": {\n ""request_id"": ""84745411...","{\n ""request_id"": ""8474541141607"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8474541141607,PRESTIGE-GEN-XII,V1,1
45,8473303/7Kcq-2024-12-09-15:27:16,8473303,2024-12-09 22:27:16+00:00,"{\n ""request"": {\n ""request_id"": ""84733037...","{\n ""request_id"": ""8473303742301"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8473303742301,PRESTIGE-GEN-XII,V1,1
50,8519243/734R-2024-12-24-01:34:49,8519243,2024-12-24 08:34:49+00:00,"{\n ""request"": {\n ""request_id"": ""85192436...","{\n ""request_id"": ""8519243666180"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8519243666180,PRESTIGE-GEN-XII,V1,1
...,...,...,...,...,...,...,...,...,...,...
4576,8598118/0bj7-2025-01-28-20:25:19,8598118,2025-01-29 03:25:19+00:00,"{\n ""request"": {\n ""request_id"": ""85981181...","{\n ""request_id"": ""8598118156230"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8598118156230,PRESTIGE-DLV1,V1,1
4577,8586464/K5Tf-2025-01-22-19:47:30,8586464,2025-01-23 02:47:30+00:00,"{\n ""request"": {\n ""request_id"": ""85864641...","{\n ""request_id"": ""8586464192537"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8586464192537,PRESTIGE-DLV1,V1,1
4578,8621427/zid2-2025-02-04-21:48:47,8621427,2025-02-05 04:48:47+00:00,"{\n ""request"": {\n ""request_id"": ""86214279...","{\n ""request_id"": ""8621427930970"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8621427930970,PRESTIGE-DLV1,V1,1
4579,8605204/BzFm-2025-01-29-20:11:00,8605204,2025-01-30 03:11:00+00:00,"{\n ""request"": {\n ""request_id"": ""86052047...","{\n ""request_id"": ""8605204732293"",\n ""rows"":...","{\n ""Response"": [\n {\n ""CounterOffer...",8605204732293,PRESTIGE-DLV1,V1,1


#### Close connection

In [13]:
conn.close()

#### Write to s3

In [14]:
%%time

str_filename = 'df_payloads.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_subtask}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 1min 54s, sys: 17.8 s, total: 2min 12s
Wall time: 2min 10s
